In [1]:
import os
import numpy as np
import pandas as pd
import osmnx as ox
from scipy.sparse import load_npz
from scipy.sparse.csgraph import dijkstra as scipy_dijkstra
from scipy.spatial import cKDTree

# 1. Nạp đồ thị & Tính trường thời gian T (phút)
adj_matrix = load_npz("data/hcm_center_moto_adj.npz")
coords = np.load("data/hcm_center_moto_meta.npz")["coords"]
tree = cKDTree(coords)

people_gps = {
    "An": (10.7725, 106.6980),
    "Bình": (10.7827, 106.6959),
    "Chi": (10.8013, 106.7114),
    "Phi": (10.7995, 106.6922),
}
sources = [int(tree.query(gps)[1]) for gps in people_gps.values()]
k = len(sources)
T = scipy_dijkstra(csgraph=adj_matrix, directed=True, indices=sources) / 60.0

# 2. Tải danh sách POI thật (Quán cà phê, nhà hàng) từ OSM và lưu cache CSV
poi_cache_path = "data/hcm_center_pois.csv"

if not os.path.exists(poi_cache_path):
    print("Đang tải dữ liệu quán cà phê & nhà hàng từ OpenStreetMap...")
    places = [
        "District 1, Ho Chi Minh City, Vietnam",
        "District 3, Ho Chi Minh City, Vietnam",
        "Binh Thanh District, Ho Chi Minh City, Vietnam"
    ]
    tags = {"amenity": ["cafe", "restaurant", "fast_food"]}
    gdf_pois = ox.features_from_place(places, tags=tags)
    
    # Lọc các địa điểm có tên rõ ràng
    gdf_pois = gdf_pois[gdf_pois["name"].notna()].copy()
    
    # Chuyển polygon (nếu có) về tọa độ tâm (centroid)
    # Chiếu sang hệ mét EPSG:32648 (UTM zone 48N - Việt Nam) để lấy tâm chuẩn rồi đổi lại lat/lng
    centroids = gdf_pois.to_crs(epsg=32648).geometry.centroid.to_crs(epsg=4326)
    gdf_pois["lat"] = centroids.y
    gdf_pois["lng"] = centroids.x
    
    # Giữ lại các cột cần thiết và lưu file CSV
    cols_to_keep = ["name", "amenity", "lat", "lng"]
    if "addr:street" in gdf_pois.columns:
        cols_to_keep.append("addr:street")
    df_pois = pd.DataFrame(gdf_pois[cols_to_keep]).drop_duplicates(subset=["name", "lat", "lng"])
    df_pois.to_csv(poi_cache_path, index=False, encoding="utf-8-sig")
    print(f"Đã tải và lưu {len(df_pois):,} địa điểm thật vào {poi_cache_path}")
else:
    df_pois = pd.read_csv(poi_cache_path)
    print(f"Đã nạp {len(df_pois):,} địa điểm thật từ cache {poi_cache_path}")

# 3. Snap mỗi POI về đỉnh đường gần nhất trên đồ thị
poi_coords = df_pois[["lat", "lng"]].values
dists_deg, nearest_nodes = tree.query(poi_coords)
df_pois["node_idx"] = nearest_nodes
df_pois["snap_dist_m"] = dists_deg * 110500.0

# Loại bỏ các POI nằm quá xa đường (> 150m)
df_pois = df_pois[df_pois["snap_dist_m"] <= 150.0].copy()

# 4. Tính thời gian di chuyển của từng người tới từng quán
poi_nodes = df_pois["node_idx"].values
T_pois = T[:, poi_nodes]  # Shape (4, num_pois)

for i, name in enumerate(people_gps.keys()):
    df_pois[f"time_{name}"] = np.round(T_pois[i], 1)

df_pois["F_sum"] = np.round(np.sum(T_pois, axis=0), 1)
df_pois["F_max"] = np.round(np.max(T_pois, axis=0), 1)

# 5. Lọc cứng & Xếp hạng theo thanh trượt cân bằng (alpha = 0.5: 50% tổng thời gian, 50% công bằng)
alpha = 0.5
df_pois["score_cost"] = (1 - alpha) * (df_pois["F_sum"] / k) + alpha * df_pois["F_max"]

# Lọc chỉ lấy quán cà phê ("cafe") và mọi người đi dưới 15 phút
cafes = df_pois[(df_pois["amenity"] == "cafe") & (df_pois["F_max"] <= 15.0)].sort_values("score_cost")

print(f"\n=== TOP 10 QUÁN CÀ PHÊ TỐI ƯU NHẤT CHO NHÓM (alpha = {alpha}) ===")
display_cols = ["name", "time_An", "time_Bình", "time_Chi", "time_Phi", "F_sum", "F_max", "score_cost"]
print(cafes[display_cols].head(10).to_string(index=False))

Đang tải dữ liệu quán cà phê & nhà hàng từ OpenStreetMap...
Đã tải và lưu 1,392 địa điểm thật vào data/hcm_center_pois.csv

=== TOP 10 QUÁN CÀ PHÊ TỐI ƯU NHẤT CHO NHÓM (alpha = 0.5) ===
            name  time_An  time_Bình  time_Chi  time_Phi  F_sum  F_max  score_cost
      Cafe Thứ 7      8.7        3.5       8.7       8.8   29.7    8.8      8.1125
Highlands Coffee      8.6        3.3       8.9       8.9   29.7    8.9      8.1625
     Miến Phở Gà      8.2        3.0       9.3       9.3   29.8    9.3      8.3750
The Coffee House      8.4        0.6       9.6       9.7   28.3    9.7      8.3875
       Phuc Long      8.0        2.8       9.1       9.5   29.5    9.5      8.4375
       Phuc Long      8.0        2.8       9.1       9.5   29.5    9.5      8.4375
        Royaltea      9.5        2.8       9.4       7.8   29.5    9.5      8.4375
       hey sugar      7.9        2.6       9.6       9.6   29.8    9.6      8.5250
    Cà Phê Cy Cy      9.6        2.9       9.5       7.9   29.8    